# Emotion Concept in OSS-LLMs

## Step 0 - Setup

Initialize runtime, settings, and project directories.

In [ ]:
import os
import sys
from huggingface_hub import login

KAGGLE_PARENT = "/kaggle/input/datasets/mustafamunir"
if os.path.isdir(KAGGLE_PARENT) and KAGGLE_PARENT not in sys.path:
    sys.path.append(KAGGLE_PARENT)

hf_token = input("hf_token (press Enter to skip): ").strip()
if hf_token:
    login(token=hf_token, add_to_git_credential=False)

In [ ]:
from __future__ import annotations

import json
import os
from datetime import datetime, timezone
from pathlib import Path

import torch
from emotion.config import Settings
from emotion.runtime import is_kaggle, is_notebook

SELECTED_MODEL = "qwen_4b"

FORCE_CUDA = True
cuda_available = torch.cuda.is_available()
gpu_count = torch.cuda.device_count() if cuda_available else 0

if FORCE_CUDA and not cuda_available:
    raise RuntimeError("FORCE_CUDA=True but CUDA is not available. Enable GPU runtime or set FORCE_CUDA=False.")

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")


if cuda_available and gpu_count > 1:
    device_map_value = "auto"
elif cuda_available:
    device_map_value = "cuda:0"
else:
    device_map_value = "auto"

settings = Settings(
    project_root="/kaggle/working" if is_kaggle() else ".",
    selected_model=SELECTED_MODEL,
    device_map=device_map_value,
)
project_root = settings.project_path

print(f"Model={settings.selected_model} | device_map={settings.device_map} | cuda={cuda_available}")

In [ ]:
standard_dirs = [
    settings.output_root,
    settings.data_dir,
    settings.results_dir,
    settings.plots_dir,
    settings.logs_dir,
]

for rel_dir in standard_dirs:
    (project_root / rel_dir).mkdir(parents=True, exist_ok=True)

run_metadata = {
    "run_id": settings.run_id,
    "timestamp_utc": datetime.now(timezone.utc).isoformat(),
    "execution_mode": settings.execution_mode,
    "is_notebook": is_notebook(),
    "is_kaggle": is_kaggle(),
    "models": [m.model_id for m in settings.enabled_models],
    "emotion_pairs": [f"{p.left}<->{p.right}" for p in settings.emotion_pairs],
}

pass

## Step 1 - Model Load Check

Load selected model and run a quick smoke test.

Locked targets:

- `Qwen/Qwen3-4B-Instruct-2507`
- `mistralai/Mistral-7B-Instruct-v0.3`
- `tiiuae/falcon-7b-instruct`
- `HuggingFaceH4/zephyr-7b-beta`
- `openchat/openchat_3.5`

In [ ]:
from emotion.model_loader import (
    load_model_with_fallback_dtypes,
    load_tokenizer,
    smoke_test_generation,
)

enabled = settings.enabled_models

In [ ]:
import gc

smoke_results = []
for spec in enabled:
    row = {
        "name": spec.name,
        "model_id": spec.model_id,
        "loaded": False,
        "dtype": None,
        "error": None,
        "sample": None,
    }

    model = None
    tokenizer = None
    try:
        tokenizer = load_tokenizer(spec.model_id, settings)
        model, dtype_name = load_model_with_fallback_dtypes(spec.model_id, settings)
        sample = smoke_test_generation(
            model,
            tokenizer,
            prompt="What is 1+1?",
            max_new_tokens=settings.max_new_tokens_smoke_test,
        )
        row["loaded"] = True
        row["dtype"] = dtype_name
        row["sample"] = sample
    except Exception as exc:  # noqa: PERF203
        row["error"] = str(exc)
    finally:
        del model
        del tokenizer
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    smoke_results.append(row)

smoke_results

In [ ]:
step_status = {
    "step_0_complete": True,
    "step_1_complete": True,
    "run_model_smoke_test": True,
    "smoke_results": smoke_results,
}

pass

## Step 2 - Probe Data Validation

Build pair spec and validate JSONL files.

Supported layouts:

1. Nested: `ROOT/{pair_key}/{split}/{side}.jsonl`
2. Flat: `ROOT/{pair_key}__{split}__{side}.jsonl`

In [ ]:
from emotion.probe_design import build_probe_spec, validate_pair_datasets

probe_spec = build_probe_spec(settings.emotion_pairs)

pass

In [ ]:
PROBE_DATASET_ROOT = "/kaggle/input/datasets/mustafamunir/emotion-probe-files"
PROBE_SPLITS = ["train"]

dataset_root = Path(PROBE_DATASET_ROOT)
pass

In [ ]:
from pathlib import Path
import json

def resolve_path(root: Path, pair_key: str, split: str, side: str):
    nested = root / pair_key / split / f"{side}.jsonl"
    flat = root / f"{pair_key}__{split}__{side}.jsonl"
    if nested.exists():
        return nested
    if flat.exists():
        return flat
    return None, nested, flat

errors = []
stats = []

for pair in settings.emotion_pairs:
    pair_key = f"{pair.left}_vs_{pair.right}"
    for split in PROBE_SPLITS:
        for side in [pair.left, pair.right]:
            res = resolve_path(dataset_root, pair_key, split, side)
            if isinstance(res, tuple) and len(res) == 3:
                _, nested, flat = res
                errors.append(
                    f"Missing file for {pair_key}/{split}/{side}. Tried: {nested} and {flat}"
                )
                continue

            path = res
            lines = path.read_text(encoding="utf-8").splitlines()
            rows = [json.loads(x) for x in lines if x.strip()]

            for i, r in enumerate(rows, 1):
                if "id" not in r or "text" not in r:
                    errors.append(f"{path}: row {i} missing id/text")
            
            lengths = [len(r["text"]) for r in rows if isinstance(r.get("text"), str)]
            stats.append({
                "pair_key": pair_key,
                "split": split,
                "side": side,
                "path": str(path),
                "n_records": len(rows),
                "avg_chars": sum(lengths)/len(lengths) if lengths else 0,
                "min_chars": min(lengths) if lengths else 0,
                "max_chars": max(lengths) if lengths else 0,
            })

step2_report = {
    "dataset_root": str(dataset_root),
    "validated": len(errors) == 0,
    "errors": errors,
    "stats": stats,
}
step2_report

In [ ]:
step_status["step_2_complete"] = bool(step2_report.get("validated", False))

pass

## Step 3 - Residual Extraction

Extract residuals for all layers at the first generated token and save artifacts.

In [ ]:
try:
    from emotion.residuals import (
        empty_cache,
        get_residuals_batched,
        load_side_jsonl,
        save_residual_artifact,
    )
    print("Using emotion.residuals from dataset package")
except Exception as _import_err:
    print("Falling back to local Step-3 helpers due to import error:")
    print(_import_err)

    import gc
    import json
    import torch

    def _resolve_path(root: Path, pair_key: str, split: str, side: str):
        nested = root / pair_key / split / f"{side}.jsonl"
        flat = root / f"{pair_key}__{split}__{side}.jsonl"
        if nested.exists():
            return nested
        if flat.exists():
            return flat
        raise FileNotFoundError(f"Missing {pair_key}/{split}/{side}. Tried: {nested} and {flat}")

    def _batchify(items, batch_size):
        return [items[i : i + batch_size] for i in range(0, len(items), batch_size)]

    def empty_cache():
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        elif torch.backends.mps.is_available():
            torch.mps.empty_cache()
        gc.collect()

    def _chat_prompts(texts, tokenizer, system_prompt):
        chats = [
            [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": t},
            ]
            for t in texts
        ]
        return list(tokenizer.apply_chat_template(chats, add_generation_prompt=True, tokenize=False))

    @torch.no_grad()
    def _get_residuals(texts, tokenizer, model, system_prompt="You are a helpful assistant."):
        prompts = _chat_prompts(texts, tokenizer, system_prompt)
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            return_token_type_ids=False,
        ).to(model.device)
        outputs = model.generate(
            **inputs,
            max_new_tokens=1,
            output_hidden_states=True,
            return_dict_in_generate=True,
            do_sample=False,
            pad_token_id=tokenizer.pad_token_id,
        )
        if not hasattr(outputs, "hidden_states") or outputs.hidden_states is None:
            raise RuntimeError("No hidden_states from generate(); cannot extract residuals.")
        hidden_states = outputs.hidden_states[0]
        residuals = torch.stack([layer[:, -1, :] for layer in hidden_states], dim=1)
        return residuals.to(torch.float32)

    @torch.no_grad()
    def get_residuals_batched(*, texts, tokenizer, model, batch_size, system_prompt="You are a helpful assistant."):
        batches = _batchify(texts, batch_size)
        chunks = [_get_residuals(batch, tokenizer, model, system_prompt) for batch in batches]
        return torch.cat(chunks, dim=0) if chunks else torch.empty(0)

    def load_side_jsonl(*, dataset_root, pair_key, split, side, max_rows=None):
        p = _resolve_path(dataset_root, pair_key, split, side)
        rows = [json.loads(x) for x in p.read_text(encoding="utf-8").splitlines() if x.strip()]
        for i, r in enumerate(rows, 1):
            if "id" not in r or "text" not in r:
                raise ValueError(f"{p}: row {i} missing id/text")
        if max_rows is not None:
            rows = rows[:max_rows]
        ids = [str(r["id"]) for r in rows]
        texts = [str(r["text"]) for r in rows]
        return ids, texts, p

    def save_residual_artifact(*, output_file, ids, texts, residuals, meta):
        output_file.parent.mkdir(parents=True, exist_ok=True)
        torch.save(
            {
                "ids": ids,
                "texts": texts,
                "residuals": residuals.cpu(),
                "meta": meta,
            },
            output_file,
        )

STEP3_BATCH_SIZE = 4
STEP3_MAX_ROWS_PER_SIDE = None
STEP3_SYSTEM_PROMPT = "You are a helpful assistant."
STEP3_CLEAR_CACHE_EACH_SIDE = True
TOKEN_POLICY = "first_generated_token"
LAYER_POLICY = "all_layers"

pass

In [ ]:
step3_report = {
    "validated_input": bool(step2_report.get("validated", False)),
    "ran_extraction": False,
    "token_policy": TOKEN_POLICY,
    "layer_policy": LAYER_POLICY,
    "batch_size": STEP3_BATCH_SIZE,
    "max_rows_per_side": STEP3_MAX_ROWS_PER_SIDE,
    "model": None,
    "dtype": None,
    "errors": [],
    "artifacts": [],
}

if not step2_report.get("validated", False):
    step3_report["errors"].append("Step 2 dataset validation failed. Fix dataset before extraction.")
elif len(enabled) != 1:
    step3_report["errors"].append(
        f"Expected exactly one selected model, got {len(enabled)}."
    )
else:
    spec = enabled[0]
    step3_report["model"] = {"name": spec.name, "model_id": spec.model_id}

step3_report

In [ ]:
if not step3_report["errors"]:
    import gc

    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    spec = enabled[0]
    tokenizer = None
    model = None

    try:
        tokenizer = load_tokenizer(spec.model_id, settings)
        model, dtype_name = load_model_with_fallback_dtypes(spec.model_id, settings)
        step3_report["dtype"] = dtype_name

        for pair in settings.emotion_pairs:
            pair_key = f"{pair.left}_vs_{pair.right}"
            for split in PROBE_SPLITS:
                for side in [pair.left, pair.right]:
                    residuals = None
                    try:
                        ids, texts, src_path = load_side_jsonl(
                            dataset_root=dataset_root,
                            pair_key=pair_key,
                            split=split,
                            side=side,
                            max_rows=STEP3_MAX_ROWS_PER_SIDE,
                        )

                        residuals = get_residuals_batched(
                            texts=texts,
                            tokenizer=tokenizer,
                            model=model,
                            batch_size=STEP3_BATCH_SIZE,
                            system_prompt=STEP3_SYSTEM_PROMPT,
                        )

                        out_file = (
                            project_root
                            / settings.results_dir
                            / "residuals"
                            / spec.name
                            / pair_key
                            / split
                            / f"{side}.pt"
                        )

                        save_residual_artifact(
                            output_file=out_file,
                            ids=ids,
                            texts=texts,
                            residuals=residuals,
                            meta={
                                "model_name": spec.name,
                                "model_id": spec.model_id,
                                "dtype": dtype_name,
                                "pair_key": pair_key,
                                "split": split,
                                "side": side,
                                "source_jsonl": str(src_path),
                                "token_policy": TOKEN_POLICY,
                                "layer_policy": LAYER_POLICY,
                                "residual_shape": list(residuals.shape),
                            },
                        )

                        step3_report["artifacts"].append(
                            {
                                "pair_key": pair_key,
                                "split": split,
                                "side": side,
                                "output_file": str(out_file),
                                "n_prompts": len(ids),
                                "residual_shape": list(residuals.shape),
                            }
                        )
                    except Exception as exc:  # noqa: PERF203
                        step3_report["errors"].append(
                            f"{pair_key}/{split}/{side}: {exc}"
                        )
                    finally:
                        if residuals is not None:
                            del residuals
                        gc.collect()
                        if STEP3_CLEAR_CACHE_EACH_SIDE and torch.cuda.is_available():
                            torch.cuda.empty_cache()

        step3_report["ran_extraction"] = True
    finally:
        if model is not None:
            del model
        if tokenizer is not None:
            del tokenizer
        empty_cache()

step3_report

In [ ]:
step_status["step_3_complete"] = (
    bool(step3_report.get("ran_extraction")) and not step3_report.get("errors")
)

pass

## Step 4 - Probe Build and Scores

Build pair probes, compute scores, and derive baseline percentages.

In [ ]:
try:
    from emotion.detection import (
        build_pair_probe_from_residuals,
        load_residual_artifact,
        pair_percentages,
        pair_score,
        save_probe_artifact,
        triggered_side,
    )

    import torch

    _labels_test, _margin_test = triggered_side(
        p_left=torch.tensor([0.9]),
        p_right=torch.tensor([0.1]),
        left_label="left",
        right_label="right",
    )
    if _labels_test != ["left"] or _margin_test.shape != torch.Size([1]):
        raise RuntimeError("emotion.detection compatibility check failed")

    print("Using emotion.detection from dataset package")
except Exception as _import_err:
    print("Falling back to local Step-4 helpers due to import error:")
    print(_import_err)

    import torch
    import torch.nn.functional as F

    def load_residual_artifact(path):
        payload = torch.load(path, map_location="cpu")
        if not isinstance(payload, dict) or "residuals" not in payload:
            raise ValueError(f"{path} is not a valid residual artifact.")
        residuals = payload["residuals"]
        if not isinstance(residuals, torch.Tensor) or residuals.ndim != 3:
            raise ValueError(f"{path} residuals must be shape (prompt, layer, hidden).")
        return payload

    def _normalize_last_dim(x, eps=1e-8):
        norm = x.norm(dim=-1, keepdim=True).clamp_min(eps)
        return x / norm

    def build_pair_probe_from_residuals(
        *,
        left_residuals,
        right_residuals,
        left_label,
        right_label,
        pair_key,
        split,
        model_name,
        model_id,
        dtype_name,
        token_policy,
        layer_policy,
        eps=1e-8,
    ):
        if left_residuals.ndim != 3 or right_residuals.ndim != 3:
            raise ValueError("Both residual tensors must be shape (prompt, layer, hidden).")
        if left_residuals.shape[1:] != right_residuals.shape[1:]:
            raise ValueError(
                f"Shape mismatch: {tuple(left_residuals.shape)} vs {tuple(right_residuals.shape)}"
            )

        left_mean = left_residuals.to(torch.float32).mean(dim=0)
        right_mean = right_residuals.to(torch.float32).mean(dim=0)
        direction_raw = left_mean - right_mean
        direction = _normalize_last_dim(direction_raw, eps=eps)

        per_layer_norm = direction_raw.norm(dim=-1)
        return {
            "probe_direction": direction.cpu(),
            "probe_direction_raw": direction_raw.cpu(),
            "left_mean": left_mean.cpu(),
            "right_mean": right_mean.cpu(),
            "meta": {
                "pair_key": pair_key,
                "left_label": left_label,
                "right_label": right_label,
                "split": split,
                "model_name": model_name,
                "model_id": model_id,
                "dtype": dtype_name,
                "token_policy": token_policy,
                "layer_policy": layer_policy,
                "left_n": int(left_residuals.shape[0]),
                "right_n": int(right_residuals.shape[0]),
                "n_layers": int(left_residuals.shape[1]),
                "hidden_size": int(left_residuals.shape[2]),
                "degenerate_layer_count": int((per_layer_norm <= eps).sum().item()),
                "min_raw_norm": float(per_layer_norm.min().item()),
                "max_raw_norm": float(per_layer_norm.max().item()),
                "mean_raw_norm": float(per_layer_norm.mean().item()),
            },
        }

    def save_probe_artifact(*, output_file, artifact):
        output_file.parent.mkdir(parents=True, exist_ok=True)
        torch.save(artifact, output_file)

    def pair_score(*, residuals, probe_direction, layer_reduction="mean", eps=1e-8):
        if residuals.ndim != 3 or probe_direction.ndim != 2:
            raise ValueError("Expected residuals=(prompt, layer, hidden), probe_direction=(layer, hidden)")
        if residuals.shape[1:] != probe_direction.shape:
            raise ValueError(
                f"Shape mismatch for scoring: {tuple(residuals.shape)} vs {tuple(probe_direction.shape)}"
            )
        if layer_reduction != "mean":
            raise ValueError(f"Unsupported layer_reduction: {layer_reduction}")

        x = F.normalize(residuals.to(torch.float32), p=2, dim=-1, eps=eps)
        v = F.normalize(probe_direction.to(torch.float32), p=2, dim=-1, eps=eps)
        per_layer = (x * v.unsqueeze(0)).sum(dim=-1)
        return per_layer.mean(dim=1)

    def pair_percentages(scores, *, temperature=1.0):
        if temperature <= 0:
            raise ValueError("temperature must be > 0")
        p_left = torch.sigmoid(temperature * scores.to(torch.float32))
        p_right = 1.0 - p_left
        return p_left, p_right

    def triggered_side(*, p_left, p_right, left_label, right_label):
        if p_left.shape != p_right.shape:
            raise ValueError("p_left and p_right shape mismatch")
        labels = [left_label if bool(m) else right_label for m in (p_left >= p_right).tolist()]
        margin = (p_left - p_right).abs()
        return labels, margin

import torch

def _safe_triggered_side(*, p_left, p_right, left_label, right_label):
    if p_left.shape != p_right.shape:
        raise ValueError("p_left and p_right shape mismatch")
    labels = [left_label if bool(m) else right_label for m in (p_left >= p_right).tolist()]
    margin = (p_left - p_right).abs()
    return labels, margin

triggered_side = _safe_triggered_side

In [ ]:
STEP4_PROBE_SPLIT = "train"
STEP4_LAYER_REDUCTION = "mean"
STEP4_SCORE_TEMPERATURE = 1.0

pass

In [ ]:
step4_report = {
    "validated_input": bool(step_status.get("step_3_complete", False)),
    "ran_probe_build": False,
    "split": STEP4_PROBE_SPLIT,
    "layer_reduction": STEP4_LAYER_REDUCTION,
    "score_temperature": STEP4_SCORE_TEMPERATURE,
    "model": None,
    "errors": [],
    "probe_artifacts": [],
    "measurements": [],
}

if not step_status.get("step_3_complete", False):
    step4_report["errors"].append("Step 3 is incomplete. Run Step 3 residual extraction first.")
elif len(enabled) != 1:
    step4_report["errors"].append(f"Expected exactly one selected model, got {len(enabled)}")
else:
    spec = enabled[0]
    step4_report["model"] = {"name": spec.name, "model_id": spec.model_id}

if not step4_report["errors"]:
    spec = enabled[0]

    for pair in settings.emotion_pairs:
        pair_key = f"{pair.left}_vs_{pair.right}"
        left = pair.left
        right = pair.right

        try:
            left_file = (
                project_root
                / settings.results_dir
                / "residuals"
                / spec.name
                / pair_key
                / STEP4_PROBE_SPLIT
                / f"{left}.pt"
            )
            right_file = (
                project_root
                / settings.results_dir
                / "residuals"
                / spec.name
                / pair_key
                / STEP4_PROBE_SPLIT
                / f"{right}.pt"
            )

            if not left_file.exists() or not right_file.exists():
                raise FileNotFoundError(
                    f"Missing residual artifacts for {pair_key}/{STEP4_PROBE_SPLIT}. "
                    f"Expected: {left_file} and {right_file}"
                )

            left_payload = load_residual_artifact(left_file)
            right_payload = load_residual_artifact(right_file)
            left_residuals = left_payload["residuals"]
            right_residuals = right_payload["residuals"]

            probe_artifact = build_pair_probe_from_residuals(
                left_residuals=left_residuals,
                right_residuals=right_residuals,
                left_label=left,
                right_label=right,
                pair_key=pair_key,
                split=STEP4_PROBE_SPLIT,
                model_name=spec.name,
                model_id=spec.model_id,
                dtype_name=step3_report.get("dtype"),
                token_policy=TOKEN_POLICY,
                layer_policy=LAYER_POLICY,
            )

            probe_out = (
                project_root
                / settings.results_dir
                / "probes"
                / spec.name
                / pair_key
                / STEP4_PROBE_SPLIT
                / "probe.pt"
            )
            save_probe_artifact(output_file=probe_out, artifact=probe_artifact)

            direction = probe_artifact["probe_direction"]
            s_left = pair_score(
                residuals=left_residuals,
                probe_direction=direction,
                layer_reduction=STEP4_LAYER_REDUCTION,
            )
            s_right = pair_score(
                residuals=right_residuals,
                probe_direction=direction,
                layer_reduction=STEP4_LAYER_REDUCTION,
            )

            left_p_left, left_p_right = pair_percentages(s_left, temperature=STEP4_SCORE_TEMPERATURE)
            right_p_left, right_p_right = pair_percentages(s_right, temperature=STEP4_SCORE_TEMPERATURE)

            left_labels, left_margin = triggered_side(
                p_left=left_p_left,
                p_right=left_p_right,
                left_label=left,
                right_label=right,
            )
            right_labels, right_margin = triggered_side(
                p_left=right_p_left,
                p_right=right_p_right,
                left_label=left,
                right_label=right,
            )

            measure = {
                "pair_key": pair_key,
                "split": STEP4_PROBE_SPLIT,
                "probe_file": str(probe_out),
                "left_source": str(left_file),
                "right_source": str(right_file),
                "left_n": int(left_residuals.shape[0]),
                "right_n": int(right_residuals.shape[0]),
                "direction_shape": list(direction.shape),
                "left_mean_score": float(s_left.mean().item()),
                "right_mean_score": float(s_right.mean().item()),
                "left_expected_side_rate": float((left_p_left > 0.5).float().mean().item()),
                "right_expected_side_rate": float((right_p_right > 0.5).float().mean().item()),
                "left_avg_margin": float(left_margin.mean().item()),
                "right_avg_margin": float(right_margin.mean().item()),
                "left_trigger_counts": {
                    left: int(sum(1 for x in left_labels if x == left)),
                    right: int(sum(1 for x in left_labels if x == right)),
                },
                "right_trigger_counts": {
                    left: int(sum(1 for x in right_labels if x == left)),
                    right: int(sum(1 for x in right_labels if x == right)),
                },
                "notes": "Uncalibrated percentages (temperature=1.0). Step 6 will calibrate.",
            }

            step4_report["probe_artifacts"].append(
                {
                    "pair_key": pair_key,
                    "probe_file": str(probe_out),
                    "meta": probe_artifact["meta"],
                }
            )
            step4_report["measurements"].append(
                {
                    "pair_key": pair_key,
                    "left_mean_score": measure["left_mean_score"],
                    "right_mean_score": measure["right_mean_score"],
                    "left_expected_side_rate": measure["left_expected_side_rate"],
                    "right_expected_side_rate": measure["right_expected_side_rate"],
                }
            )

        except Exception as exc:  # noqa: PERF203
            step4_report["errors"].append(f"{pair_key}/{STEP4_PROBE_SPLIT}: {exc}")

    step4_report["ran_probe_build"] = True

step4_report

In [ ]:
step_status["step_4_complete"] = (
    bool(step4_report.get("ran_probe_build")) and not step4_report.get("errors")
)

pass

## Step 5 - Layer Policy

Sweep layers, pick best layer/band, and store global policy.

In [ ]:
try:
    from emotion.layer_policy import (
        best_contiguous_band,
        best_single_layer_index,
        choose_global_layer_with_tiebreak,
        evaluate_policy_accuracy,
        per_layer_pair_scores,
        summarize_layer_sweep,
    )
    print("Using emotion.layer_policy from dataset package")
except Exception as _import_err:
    print("Falling back to local Step-5 helpers due to import error:")
    print(_import_err)

    import torch
    import torch.nn.functional as F

    def per_layer_pair_scores(*, residuals, probe_direction, eps=1e-8):
        if residuals.ndim != 3 or probe_direction.ndim != 2:
            raise ValueError("Expected residuals=(prompt, layer, hidden), probe_direction=(layer, hidden)")
        if residuals.shape[1:] != probe_direction.shape:
            raise ValueError(
                f"Shape mismatch: {tuple(residuals.shape)} vs {tuple(probe_direction.shape)}"
            )
        x = F.normalize(residuals.to(torch.float32), p=2, dim=-1, eps=eps)
        v = F.normalize(probe_direction.to(torch.float32), p=2, dim=-1, eps=eps)
        return (x * v.unsqueeze(0)).sum(dim=-1)

    class _LayerSweepSummary:
        def __init__(self, per_layer_accuracy, per_layer_separation, left_positive_rate, right_negative_rate):
            self.per_layer_accuracy = per_layer_accuracy
            self.per_layer_separation = per_layer_separation
            self.left_positive_rate = left_positive_rate
            self.right_negative_rate = right_negative_rate

    def summarize_layer_sweep(*, left_scores, right_scores):
        if left_scores.ndim != 2 or right_scores.ndim != 2:
            raise ValueError("left_scores/right_scores must be shape (prompt, layer)")
        if left_scores.shape[1] != right_scores.shape[1]:
            raise ValueError("left_scores and right_scores must have same n_layers")
        left_positive_rate = (left_scores > 0).float().mean(dim=0)
        right_negative_rate = (right_scores < 0).float().mean(dim=0)
        per_layer_accuracy = 0.5 * (left_positive_rate + right_negative_rate)
        per_layer_separation = left_scores.mean(dim=0) - right_scores.mean(dim=0)
        return _LayerSweepSummary(
            per_layer_accuracy=per_layer_accuracy,
            per_layer_separation=per_layer_separation,
            left_positive_rate=left_positive_rate,
            right_negative_rate=right_negative_rate,
        )

    def best_single_layer_index(per_layer_accuracy):
        return int(torch.argmax(per_layer_accuracy).item())

    def best_contiguous_band(*, per_layer_accuracy, band_width):
        n_layers = int(per_layer_accuracy.shape[0])
        width = min(int(band_width), n_layers)
        best_start = 0
        best_mean = float("-inf")
        for start in range(0, n_layers - width + 1):
            end = start + width
            mean_acc = float(per_layer_accuracy[start:end].mean().item())
            if mean_acc > best_mean:
                best_mean = mean_acc
                best_start = start
        best_end = best_start + width - 1
        return {
            "start": best_start,
            "end": best_end,
            "width": width,
            "mean_accuracy": best_mean,
        }

    def evaluate_policy_accuracy(*, left_scores, right_scores, selected_layers):
        idx = torch.tensor(selected_layers, dtype=torch.long)
        left_agg = left_scores.index_select(dim=1, index=idx).mean(dim=1)
        right_agg = right_scores.index_select(dim=1, index=idx).mean(dim=1)
        left_rate = float((left_agg > 0).float().mean().item())
        right_rate = float((right_agg < 0).float().mean().item())
        return {
            "left_expected_side_rate": left_rate,
            "right_expected_side_rate": right_rate,
            "balanced_accuracy": 0.5 * (left_rate + right_rate),
            "mean_separation": float(left_agg.mean().item() - right_agg.mean().item()),
        }

    def choose_global_layer_with_tiebreak(*, layer_votes, layer_stats):
        ranking = sorted(
            layer_votes.keys(),
            key=lambda layer: (
                int(layer_votes[layer]),
                float(layer_stats[layer]["avg_accuracy"]),
                float(layer_stats[layer]["avg_separation"]),
                -int(layer),
            ),
            reverse=True,
        )
        winner = int(ranking[0])
        return {
            "winner": winner,
            "ranking": ranking,
            "winner_stats": {
                "votes": int(layer_votes[winner]),
                "avg_accuracy": float(layer_stats[winner]["avg_accuracy"]),
                "avg_separation": float(layer_stats[winner]["avg_separation"]),
            },
        }

In [ ]:
STEP5_POLICY_SPLIT = "train"
STEP5_BAND_WIDTH = 4

pass

In [ ]:
step5_report = {
    "validated_input": bool(step_status.get("step_4_complete", False)),
    "ran_layer_sweep": False,
    "split": STEP5_POLICY_SPLIT,
    "band_width": STEP5_BAND_WIDTH,
    "model": None,
    "errors": [],
    "pair_summaries": [],
    "global_policy": None,
}

if not step_status.get("step_4_complete", False):
    step5_report["errors"].append("Step 4 is incomplete. Run Step 4 first.")
elif len(enabled) != 1:
    step5_report["errors"].append(f"Expected exactly one selected model, got {len(enabled)}")
else:
    spec = enabled[0]
    step5_report["model"] = {"name": spec.name, "model_id": spec.model_id}

if not step5_report["errors"]:
    import torch

    spec = enabled[0]
    pair_top_layers = []
    pair_layer_stats = {}
    n_layers_seen = 0

    for pair in settings.emotion_pairs:
        pair_key = f"{pair.left}_vs_{pair.right}"
        left = pair.left
        right = pair.right

        try:
            left_res_file = (
                project_root
                / settings.results_dir
                / "residuals"
                / spec.name
                / pair_key
                / STEP5_POLICY_SPLIT
                / f"{left}.pt"
            )
            right_res_file = (
                project_root
                / settings.results_dir
                / "residuals"
                / spec.name
                / pair_key
                / STEP5_POLICY_SPLIT
                / f"{right}.pt"
            )
            probe_file = (
                project_root
                / settings.results_dir
                / "probes"
                / spec.name
                / pair_key
                / STEP5_POLICY_SPLIT
                / "probe.pt"
            )

            if not left_res_file.exists() or not right_res_file.exists() or not probe_file.exists():
                raise FileNotFoundError(
                    f"Missing Step-5 inputs for {pair_key}/{STEP5_POLICY_SPLIT}. "
                    f"Expected residuals ({left_res_file}, {right_res_file}) and probe ({probe_file})."
                )

            left_payload = torch.load(left_res_file, map_location="cpu")
            right_payload = torch.load(right_res_file, map_location="cpu")
            probe_payload = torch.load(probe_file, map_location="cpu")

            left_residuals = left_payload["residuals"]
            right_residuals = right_payload["residuals"]
            probe_direction = probe_payload["probe_direction"]

            left_scores = per_layer_pair_scores(
                residuals=left_residuals,
                probe_direction=probe_direction,
            )
            right_scores = per_layer_pair_scores(
                residuals=right_residuals,
                probe_direction=probe_direction,
            )
            sweep = summarize_layer_sweep(left_scores=left_scores, right_scores=right_scores)
            n_layers_seen = max(n_layers_seen, int(sweep.per_layer_accuracy.shape[0]))

            best_layer = best_single_layer_index(sweep.per_layer_accuracy)
            pair_top_layers.append(best_layer)
            best_layer_accuracy = float(sweep.per_layer_accuracy[best_layer].item())
            best_layer_separation = float(sweep.per_layer_separation[best_layer].item())
            pair_layer_stats[pair_key] = {
                "layer": int(best_layer),
                "accuracy": best_layer_accuracy,
                "separation": best_layer_separation,
            }

            best_band = best_contiguous_band(
                per_layer_accuracy=sweep.per_layer_accuracy,
                band_width=STEP5_BAND_WIDTH,
            )
            band_layers = list(range(best_band["start"], best_band["end"] + 1))

            single_metrics = evaluate_policy_accuracy(
                left_scores=left_scores,
                right_scores=right_scores,
                selected_layers=[best_layer],
            )
            band_metrics = evaluate_policy_accuracy(
                left_scores=left_scores,
                right_scores=right_scores,
                selected_layers=band_layers,
            )

            pair_summary = {
                "pair_key": pair_key,
                "split": STEP5_POLICY_SPLIT,
                "n_layers": int(sweep.per_layer_accuracy.shape[0]),
                "best_single_layer": best_layer,
                "best_single_layer_accuracy": best_layer_accuracy,
                "best_single_layer_separation": best_layer_separation,
                "best_band": best_band,
                "single_layer_metrics": single_metrics,
                "band_metrics": band_metrics,
                "top5_layers_by_accuracy": [
                    int(i)
                    for i in torch.argsort(sweep.per_layer_accuracy, descending=True)[:5].tolist()
                ],
                "per_layer_accuracy": [float(x) for x in sweep.per_layer_accuracy.tolist()],
                "per_layer_separation": [float(x) for x in sweep.per_layer_separation.tolist()],
            }

            pair_policy_path = (
                project_root
                / settings.results_dir
                / "layer_policy"
                / spec.name
                / pair_key
                / STEP5_POLICY_SPLIT
                / "layer_policy.json"
            )
            pair_policy_path.parent.mkdir(parents=True, exist_ok=True)
            pair_policy_path.write_text(json.dumps(pair_summary, indent=2), encoding="utf-8")

            step5_report["pair_summaries"].append(
                {
                    "pair_key": pair_key,
                    "policy_file": str(pair_policy_path),
                    "best_single_layer": best_layer,
                    "best_single_layer_accuracy": pair_summary["best_single_layer_accuracy"],
                    "best_band": best_band,
                }
            )

        except Exception as exc:  # noqa: PERF203
            step5_report["errors"].append(f"{pair_key}/{STEP5_POLICY_SPLIT}: {exc}")

    if pair_top_layers:
        layer_votes = {}
        for layer_idx in pair_top_layers:
            layer_votes[layer_idx] = layer_votes.get(layer_idx, 0) + 1

        layer_stats_by_layer = {}
        for layer_idx in layer_votes:
            matching = [
                stats for stats in pair_layer_stats.values() if int(stats["layer"]) == int(layer_idx)
            ]
            avg_accuracy = sum(float(s["accuracy"]) for s in matching) / len(matching)
            avg_separation = sum(float(s["separation"]) for s in matching) / len(matching)
            layer_stats_by_layer[int(layer_idx)] = {
                "avg_accuracy": float(avg_accuracy),
                "avg_separation": float(avg_separation),
            }

        chooser = choose_global_layer_with_tiebreak(
            layer_votes=layer_votes,
            layer_stats=layer_stats_by_layer,
        )
        global_best_single = int(chooser["winner"])

        inferred_layers = max(1, int(n_layers_seen))
        global_band_width = min(STEP5_BAND_WIDTH, inferred_layers)
        global_start = max(0, min(global_best_single - (global_band_width // 2), inferred_layers - global_band_width))
        global_end = global_start + global_band_width - 1

        step5_report["global_policy"] = {
            "method": "majority_vote_with_quality_tiebreak",
            "tie_break_order": [
                "votes_desc",
                "avg_accuracy_desc",
                "avg_separation_desc",
                "layer_index_asc",
            ],
            "best_single_layer": int(global_best_single),
            "best_band": {
                "start": int(global_start),
                "end": int(global_end),
                "width": int(global_band_width),
            },
            "layer_votes": {str(k): int(v) for k, v in sorted(layer_votes.items())},
            "layer_stats": {
                str(k): {
                    "avg_accuracy": float(v["avg_accuracy"]),
                    "avg_separation": float(v["avg_separation"]),
                }
                for k, v in sorted(layer_stats_by_layer.items())
            },
            "ranking": [int(x) for x in chooser["ranking"]],
            "winner_stats": {
                "votes": int(chooser["winner_stats"]["votes"]),
                "avg_accuracy": float(chooser["winner_stats"]["avg_accuracy"]),
                "avg_separation": float(chooser["winner_stats"]["avg_separation"]),
            },
            "note": "Train-split policy. Recompute on val split when available.",
        }

    step5_report["ran_layer_sweep"] = True

step5_report

In [ ]:
step_status["step_5_complete"] = (
    bool(step5_report.get("ran_layer_sweep")) and not step5_report.get("errors")
)

pass

## Step 6 - Calibration

Calibrate temperature `k` for percentage mapping and log calibration quality.

In [ ]:
try:
    from emotion.calibration import (
        aggregate_scores_by_layers,
        build_labeled_pair_scores,
        sweep_temperature_grid,
    )
    from emotion.layer_policy import per_layer_pair_scores
    print("Using emotion.calibration + emotion.layer_policy from dataset package")
except Exception as _import_err:
    print("Falling back to local Step-6 helpers due to import error:")
    print(_import_err)

    import torch
    import torch.nn.functional as F

    def per_layer_pair_scores(*, residuals, probe_direction, eps=1e-8):
        if residuals.ndim != 3 or probe_direction.ndim != 2:
            raise ValueError("Expected residuals=(prompt, layer, hidden), probe_direction=(layer, hidden)")
        if residuals.shape[1:] != probe_direction.shape:
            raise ValueError(
                f"Shape mismatch: {tuple(residuals.shape)} vs {tuple(probe_direction.shape)}"
            )
        x = F.normalize(residuals.to(torch.float32), p=2, dim=-1, eps=eps)
        v = F.normalize(probe_direction.to(torch.float32), p=2, dim=-1, eps=eps)
        return (x * v.unsqueeze(0)).sum(dim=-1)

    def aggregate_scores_by_layers(*, per_layer_scores, selected_layers):
        if per_layer_scores.ndim != 2:
            raise ValueError("per_layer_scores must be shape (prompt, layer)")
        if not selected_layers:
            raise ValueError("selected_layers cannot be empty")
        idx = torch.tensor(selected_layers, dtype=torch.long)
        return per_layer_scores.index_select(dim=1, index=idx).mean(dim=1)

    def build_labeled_pair_scores(*, left_scores, right_scores):
        scores = torch.cat([left_scores, right_scores], dim=0).to(torch.float32)
        labels = torch.cat(
            [
                torch.ones_like(left_scores, dtype=torch.float32),
                torch.zeros_like(right_scores, dtype=torch.float32),
            ],
            dim=0,
        )
        return scores, labels

    def _ece(probs, labels, n_bins=10):
        p = probs.to(torch.float32).clamp(0.0, 1.0)
        y = labels.to(torch.float32)
        edges = torch.linspace(0.0, 1.0, n_bins + 1)
        total = p.numel()
        ece = 0.0
        for i in range(n_bins):
            lo = edges[i]
            hi = edges[i + 1]
            mask = (p >= lo) & (p <= hi) if i == n_bins - 1 else (p >= lo) & (p < hi)
            cnt = int(mask.sum().item())
            if cnt == 0:
                continue
            acc = float(y[mask].mean().item())
            conf = float(p[mask].mean().item())
            ece += (cnt / total) * abs(acc - conf)
        return float(ece)

    def sweep_temperature_grid(*, scores, labels, temperatures, ece_bins=10):
        best_temp = None
        best_nll = float("inf")
        best_ece = float("inf")
        best_brier = float("inf")
        rows = []
        for t in temperatures:
            logits = float(t) * scores.to(torch.float32)
            y = labels.to(torch.float32)
            nll = float(F.binary_cross_entropy_with_logits(logits, y).item())
            probs = torch.sigmoid(logits)
            brier = float(torch.mean((probs - y) ** 2).item())
            ece = _ece(probs, y, n_bins=ece_bins)
            rows.append(
                {
                    "temperature": float(t),
                    "nll": nll,
                    "brier": brier,
                    "ece": ece,
                }
            )
            if nll < best_nll or (nll == best_nll and ece < best_ece):
                best_temp = float(t)
                best_nll = float(nll)
                best_ece = float(ece)
                best_brier = float(brier)

        return {
            "best_temperature": float(best_temp),
            "best_nll": float(best_nll),
            "best_brier": float(best_brier),
            "best_ece": float(best_ece),
            "grid_metrics": rows,
        }

In [ ]:
STEP6_CALIBRATION_SPLIT = "train"
STEP6_USE_GLOBAL_POLICY = True
STEP6_ECE_BINS = 10

STEP6_TEMPERATURE_GRID = [
    0.10,
    0.20,
    0.30,
    0.40,
    0.50,
    0.75,
    1.00,
    1.25,
    1.50,
    1.75,
    2.00,
    2.50,
    3.00,
    4.00,
    5.00,
    6.00,
    8.00,
    10.00,
    12.00,
    15.00,
    20.00,
]
STEP6_AUTO_EXTEND_GRID = True
STEP6_MAX_TEMPERATURE = 40.0
STEP6_MAX_EXTENSION_ROUNDS = 4

pass

In [ ]:
step6_report = {
    "validated_input": bool(step_status.get("step_5_complete", False)),
    "ran_calibration": False,
    "split": STEP6_CALIBRATION_SPLIT,
    "use_global_policy": STEP6_USE_GLOBAL_POLICY,
    "temperature_grid": [float(x) for x in STEP6_TEMPERATURE_GRID],
    "auto_extend_grid": bool(STEP6_AUTO_EXTEND_GRID),
    "max_temperature": float(STEP6_MAX_TEMPERATURE),
    "max_extension_rounds": int(STEP6_MAX_EXTENSION_ROUNDS),
    "model": None,
    "errors": [],
    "warnings": [],
    "pair_calibration": [],
    "global_calibration": None,
}

if not step_status.get("step_5_complete", False):
    step6_report["errors"].append("Step 5 is incomplete. Run Step 5 first.")
elif len(enabled) != 1:
    step6_report["errors"].append(f"Expected exactly one selected model, got {len(enabled)}")
else:
    spec = enabled[0]
    step6_report["model"] = {"name": spec.name, "model_id": spec.model_id}

if not step6_report["errors"]:
    import torch

    spec = enabled[0]
    global_scores = []
    global_labels = []

    if STEP6_USE_GLOBAL_POLICY:
        global_policy = step5_report.get("global_policy") if isinstance(step5_report, dict) else None
        if not global_policy:
            step6_report["errors"].append("Missing step5_report.global_policy")
        else:
            gb = global_policy.get("best_band")
            if not gb:
                step6_report["errors"].append("Missing global best_band in Step 5 report")
            else:
                global_layers = list(range(int(gb["start"]), int(gb["end"]) + 1))
    else:
        global_layers = None

    def _calibrate_with_optional_extension(scores, labels):
        tested_grid = sorted(
            {
                float(t)
                for t in STEP6_TEMPERATURE_GRID
                if float(t) > 0.0 and float(t) <= float(STEP6_MAX_TEMPERATURE)
            }
        )
        if not tested_grid:
            raise ValueError("STEP6_TEMPERATURE_GRID must contain positive values.")

        rounds = 0
        sweep = sweep_temperature_grid(
            scores=scores,
            labels=labels,
            temperatures=tested_grid,
            ece_bins=STEP6_ECE_BINS,
        )

        while (
            STEP6_AUTO_EXTEND_GRID
            and rounds < STEP6_MAX_EXTENSION_ROUNDS
            and abs(float(sweep["best_temperature"]) - max(tested_grid)) < 1e-12
            and max(tested_grid) < float(STEP6_MAX_TEMPERATURE)
        ):
            current_max = max(tested_grid)
            next_candidates = [
                current_max * 1.25,
                current_max * 1.50,
                current_max * 1.75,
                min(float(STEP6_MAX_TEMPERATURE), current_max * 2.0),
            ]
            for t in next_candidates:
                t = float(min(float(STEP6_MAX_TEMPERATURE), t))
                if t > current_max:
                    tested_grid.append(t)
            tested_grid = sorted(set(tested_grid))
            rounds += 1
            sweep = sweep_temperature_grid(
                scores=scores,
                labels=labels,
                temperatures=tested_grid,
                ece_bins=STEP6_ECE_BINS,
            )

        edge_hit = abs(float(sweep["best_temperature"]) - max(tested_grid)) < 1e-12
        return sweep, tested_grid, edge_hit, rounds

    for pair in settings.emotion_pairs:
        pair_key = f"{pair.left}_vs_{pair.right}"
        left = pair.left
        right = pair.right

        try:
            left_res_file = (
                project_root
                / settings.results_dir
                / "residuals"
                / spec.name
                / pair_key
                / STEP6_CALIBRATION_SPLIT
                / f"{left}.pt"
            )
            right_res_file = (
                project_root
                / settings.results_dir
                / "residuals"
                / spec.name
                / pair_key
                / STEP6_CALIBRATION_SPLIT
                / f"{right}.pt"
            )
            probe_file = (
                project_root
                / settings.results_dir
                / "probes"
                / spec.name
                / pair_key
                / STEP6_CALIBRATION_SPLIT
                / "probe.pt"
            )
            pair_policy_file = (
                project_root
                / settings.results_dir
                / "layer_policy"
                / spec.name
                / pair_key
                / STEP6_CALIBRATION_SPLIT
                / "layer_policy.json"
            )

            if not left_res_file.exists() or not right_res_file.exists() or not probe_file.exists():
                raise FileNotFoundError(
                    f"Missing Step-6 inputs for {pair_key}/{STEP6_CALIBRATION_SPLIT}. "
                    f"Expected residuals ({left_res_file}, {right_res_file}) and probe ({probe_file})."
                )
            if not pair_policy_file.exists() and not STEP6_USE_GLOBAL_POLICY:
                raise FileNotFoundError(
                    f"Missing pair layer policy for {pair_key}/{STEP6_CALIBRATION_SPLIT}: {pair_policy_file}"
                )

            left_payload = torch.load(left_res_file, map_location="cpu")
            right_payload = torch.load(right_res_file, map_location="cpu")
            probe_payload = torch.load(probe_file, map_location="cpu")

            left_residuals = left_payload["residuals"]
            right_residuals = right_payload["residuals"]
            probe_direction = probe_payload["probe_direction"]

            left_per_layer = per_layer_pair_scores(residuals=left_residuals, probe_direction=probe_direction)
            right_per_layer = per_layer_pair_scores(residuals=right_residuals, probe_direction=probe_direction)

            if STEP6_USE_GLOBAL_POLICY:
                selected_layers = global_layers
                layer_source = "global_best_band"
            else:
                pair_policy = json.loads(pair_policy_file.read_text(encoding="utf-8"))
                bb = pair_policy["best_band"]
                selected_layers = list(range(int(bb["start"]), int(bb["end"]) + 1))
                layer_source = "pair_best_band"

            left_scores = aggregate_scores_by_layers(
                per_layer_scores=left_per_layer,
                selected_layers=selected_layers,
            )
            right_scores = aggregate_scores_by_layers(
                per_layer_scores=right_per_layer,
                selected_layers=selected_layers,
            )

            scores, labels = build_labeled_pair_scores(
                left_scores=left_scores,
                right_scores=right_scores,
            )
            sweep, tested_grid, edge_hit, extension_rounds = _calibrate_with_optional_extension(
                scores=scores,
                labels=labels,
            )

            pair_cal = {
                "pair_key": pair_key,
                "split": STEP6_CALIBRATION_SPLIT,
                "layer_source": layer_source,
                "selected_layers": [int(x) for x in selected_layers],
                "n_samples": int(scores.shape[0]),
                "best_temperature": float(sweep["best_temperature"]),
                "best_nll": float(sweep["best_nll"]),
                "best_brier": float(sweep["best_brier"]),
                "best_ece": float(sweep["best_ece"]),
                "tested_temperature_grid": [float(x) for x in tested_grid],
                "edge_hit_max_temperature": bool(edge_hit),
                "extension_rounds": int(extension_rounds),
                "grid_metrics": sweep["grid_metrics"],
            }
            if edge_hit:
                step6_report["warnings"].append(
                    f"{pair_key}: best_temperature still at tested max ({max(tested_grid)}). "
                    "Consider increasing STEP6_MAX_TEMPERATURE or using val split."
                )

            pair_cal_path = (
                project_root
                / settings.results_dir
                / "calibration"
                / spec.name
                / pair_key
                / STEP6_CALIBRATION_SPLIT
                / "temperature_calibration.json"
            )
            pair_cal_path.parent.mkdir(parents=True, exist_ok=True)
            pair_cal_path.write_text(json.dumps(pair_cal, indent=2), encoding="utf-8")

            step6_report["pair_calibration"].append(
                {
                    "pair_key": pair_key,
                    "calibration_file": str(pair_cal_path),
                    "best_temperature": float(sweep["best_temperature"]),
                    "best_nll": float(sweep["best_nll"]),
                    "best_ece": float(sweep["best_ece"]),
                    "edge_hit_max_temperature": bool(edge_hit),
                    "extension_rounds": int(extension_rounds),
                }
            )

            global_scores.append(scores)
            global_labels.append(labels)

        except Exception as exc:  # noqa: PERF203
            step6_report["errors"].append(f"{pair_key}/{STEP6_CALIBRATION_SPLIT}: {exc}")

    if global_scores and global_labels:
        scores_all = torch.cat(global_scores, dim=0)
        labels_all = torch.cat(global_labels, dim=0)
        global_sweep, global_tested_grid, global_edge_hit, global_extension_rounds = _calibrate_with_optional_extension(
            scores=scores_all,
            labels=labels_all,
        )
        step6_report["global_calibration"] = {
            "n_samples": int(scores_all.shape[0]),
            "best_temperature": float(global_sweep["best_temperature"]),
            "best_nll": float(global_sweep["best_nll"]),
            "best_brier": float(global_sweep["best_brier"]),
            "best_ece": float(global_sweep["best_ece"]),
            "tested_temperature_grid": [float(x) for x in global_tested_grid],
            "edge_hit_max_temperature": bool(global_edge_hit),
            "extension_rounds": int(global_extension_rounds),
            "grid_metrics": global_sweep["grid_metrics"],
            "note": "Use this global temperature as default k for pair percentages.",
        }
        if global_edge_hit:
            step6_report["warnings"].append(
                f"global: best_temperature still at tested max ({max(global_tested_grid)}). "
                "Consider increasing STEP6_MAX_TEMPERATURE or using val split."
            )

        global_cal_path = (
            project_root
            / settings.results_dir
            / "calibration"
            / spec.name
            / "global"
            / STEP6_CALIBRATION_SPLIT
            / "temperature_calibration.json"
        )
        global_cal_path.parent.mkdir(parents=True, exist_ok=True)
        global_cal_path.write_text(
            json.dumps(step6_report["global_calibration"], indent=2),
            encoding="utf-8",
        )
        step6_report["global_calibration"]["calibration_file"] = str(global_cal_path)

    step6_report["ran_calibration"] = True

step6_report

In [ ]:
step_status["step_6_complete"] = (
    bool(step6_report.get("ran_calibration")) and not step6_report.get("errors")
)

pass

## Step 7 - Evaluation

Run per-pair and model-level evaluation with overlap/confusion checks.

In [ ]:
try:
    from emotion.calibration import aggregate_scores_by_layers
    from emotion.evaluation import (
        evaluate_mapped_cross_confusion,
        evaluate_pair_predictions,
    )
    from emotion.layer_policy import per_layer_pair_scores
    print("Using emotion.evaluation + dependencies from dataset package")
except Exception as _import_err:
    print("Falling back to local Step-7 helpers due to import error:")
    print(_import_err)

    import torch
    import torch.nn.functional as F

    def per_layer_pair_scores(*, residuals, probe_direction, eps=1e-8):
        if residuals.ndim != 3 or probe_direction.ndim != 2:
            raise ValueError("Expected residuals=(prompt, layer, hidden), probe_direction=(layer, hidden)")
        if residuals.shape[1:] != probe_direction.shape:
            raise ValueError(
                f"Shape mismatch: {tuple(residuals.shape)} vs {tuple(probe_direction.shape)}"
            )
        x = F.normalize(residuals.to(torch.float32), p=2, dim=-1, eps=eps)
        v = F.normalize(probe_direction.to(torch.float32), p=2, dim=-1, eps=eps)
        return (x * v.unsqueeze(0)).sum(dim=-1)

    def aggregate_scores_by_layers(*, per_layer_scores, selected_layers):
        idx = torch.tensor(selected_layers, dtype=torch.long)
        return per_layer_scores.index_select(dim=1, index=idx).mean(dim=1)

    def _prob(scores, temperature):
        p_left = torch.sigmoid(float(temperature) * scores.to(torch.float32))
        p_right = 1.0 - p_left
        return p_left, p_right

    def _classify(p_left, p_right, left_label, right_label):
        labels = [left_label if bool(x) else right_label for x in (p_left >= p_right).tolist()]
        margin = (p_left - p_right).abs()
        return labels, margin

    def evaluate_pair_predictions(*, left_scores, right_scores, left_label, right_label, temperature):
        left_p_left, left_p_right = _prob(left_scores, temperature)
        right_p_left, right_p_right = _prob(right_scores, temperature)
        left_pred, left_margin = _classify(left_p_left, left_p_right, left_label, right_label)
        right_pred, right_margin = _classify(right_p_left, right_p_right, left_label, right_label)
        left_correct_rate = float(sum(1 for x in left_pred if x == left_label) / len(left_pred))
        right_correct_rate = float(sum(1 for x in right_pred if x == right_label) / len(right_pred))
        return {
            "left_correct_rate": left_correct_rate,
            "right_correct_rate": right_correct_rate,
            "balanced_accuracy": 0.5 * (left_correct_rate + right_correct_rate),
            "left_avg_margin": float(left_margin.mean().item()),
            "right_avg_margin": float(right_margin.mean().item()),
            "avg_margin": float(torch.cat([left_margin, right_margin]).mean().item()),
            "left_mean_p_left": float(left_p_left.mean().item()),
            "right_mean_p_left": float(right_p_left.mean().item()),
            "left_pred_counts": {
                left_label: int(sum(1 for x in left_pred if x == left_label)),
                right_label: int(sum(1 for x in left_pred if x == right_label)),
            },
            "right_pred_counts": {
                left_label: int(sum(1 for x in right_pred if x == left_label)),
                right_label: int(sum(1 for x in right_pred if x == right_label)),
            },
        }

    def evaluate_mapped_cross_confusion(
        *,
        source_left_scores_on_target_probe,
        source_right_scores_on_target_probe,
        target_left_label,
        target_right_label,
        expected_label_for_source_left,
        expected_label_for_source_right,
        temperature,
    ):
        left_p_left, left_p_right = _prob(source_left_scores_on_target_probe, temperature)
        right_p_left, right_p_right = _prob(source_right_scores_on_target_probe, temperature)
        left_pred, _ = _classify(left_p_left, left_p_right, target_left_label, target_right_label)
        right_pred, _ = _classify(right_p_left, right_p_right, target_left_label, target_right_label)
        left_map_acc = float(sum(1 for x in left_pred if x == expected_label_for_source_left) / len(left_pred))
        right_map_acc = float(sum(1 for x in right_pred if x == expected_label_for_source_right) / len(right_pred))
        return {
            "target_labels": [target_left_label, target_right_label],
            "expected_map": {
                "source_left_expected_target_label": expected_label_for_source_left,
                "source_right_expected_target_label": expected_label_for_source_right,
            },
            "source_left_mapped_rate": left_map_acc,
            "source_right_mapped_rate": right_map_acc,
            "mapped_accuracy": 0.5 * (left_map_acc + right_map_acc),
            "source_left_mean_p_target_left": float(left_p_left.mean().item()),
            "source_right_mean_p_target_left": float(right_p_left.mean().item()),
        }

In [ ]:
STEP7_EVAL_SPLIT = "train"
STEP7_USE_GLOBAL_POLICY = True
STEP7_TEMPERATURE_SOURCE = "global"

STEP7_OVERLAP_CHECKS = [
    {
        "source_pair": "fear_vs_confidence",
        "target_pair": "anxious_vs_relaxed",
        "expected_map": {
            "source_left_to_target": "anxious",      # fear -> anxious
            "source_right_to_target": "relaxed",     # confidence -> relaxed
        },
    },
    {
        "source_pair": "anxious_vs_relaxed",
        "target_pair": "fear_vs_confidence",
        "expected_map": {
            "source_left_to_target": "fear",         # anxious -> fear
            "source_right_to_target": "confidence",  # relaxed -> confidence
        },
    },
]

pass

In [ ]:
step7_report = {
    "validated_input": bool(step_status.get("step_6_complete", False)),
    "ran_evaluation": False,
    "split": STEP7_EVAL_SPLIT,
    "use_global_policy": STEP7_USE_GLOBAL_POLICY,
    "temperature_source": STEP7_TEMPERATURE_SOURCE,
    "model": None,
    "errors": [],
    "warnings": [],
    "pair_metrics": [],
    "aggregate_metrics": None,
    "overlap_confusion": [],
}

if not step_status.get("step_6_complete", False):
    step7_report["errors"].append("Step 6 is incomplete. Run Step 6 first.")
elif len(enabled) != 1:
    step7_report["errors"].append(f"Expected exactly one selected model, got {len(enabled)}")
else:
    spec = enabled[0]
    step7_report["model"] = {"name": spec.name, "model_id": spec.model_id}

if not step7_report["errors"]:
    import csv
    import torch

    spec = enabled[0]
    pair_lookup = {f"{p.left}_vs_{p.right}": (p.left, p.right) for p in settings.emotion_pairs}

    global_policy = step5_report.get("global_policy") if isinstance(step5_report, dict) else None
    global_cal = step6_report.get("global_calibration") if isinstance(step6_report, dict) else None

    if STEP7_USE_GLOBAL_POLICY and not global_policy:
        step7_report["errors"].append("Missing step5_report.global_policy")
    if STEP7_TEMPERATURE_SOURCE == "global" and not global_cal:
        step7_report["errors"].append("Missing step6_report.global_calibration")

    if not step7_report["errors"]:
        if STEP7_USE_GLOBAL_POLICY:
            gb = global_policy.get("best_band")
            if not gb:
                step7_report["errors"].append("Missing global best_band in Step 5 report")
            else:
                global_layers = list(range(int(gb["start"]), int(gb["end"]) + 1))
        else:
            global_layers = None

    pair_rows_for_csv = []

    if not step7_report["errors"]:
        for pair in settings.emotion_pairs:
            pair_key = f"{pair.left}_vs_{pair.right}"
            left = pair.left
            right = pair.right

            try:
                left_res_file = (
                    project_root / settings.results_dir / "residuals" / spec.name / pair_key / STEP7_EVAL_SPLIT / f"{left}.pt"
                )
                right_res_file = (
                    project_root / settings.results_dir / "residuals" / spec.name / pair_key / STEP7_EVAL_SPLIT / f"{right}.pt"
                )
                probe_file = (
                    project_root / settings.results_dir / "probes" / spec.name / pair_key / STEP7_EVAL_SPLIT / "probe.pt"
                )
                pair_policy_file = (
                    project_root / settings.results_dir / "layer_policy" / spec.name / pair_key / STEP7_EVAL_SPLIT / "layer_policy.json"
                )
                pair_cal_file = (
                    project_root / settings.results_dir / "calibration" / spec.name / pair_key / STEP7_EVAL_SPLIT / "temperature_calibration.json"
                )

                if not left_res_file.exists() or not right_res_file.exists() or not probe_file.exists():
                    raise FileNotFoundError(
                        f"Missing Step-7 inputs for {pair_key}/{STEP7_EVAL_SPLIT}. "
                        f"Expected residuals ({left_res_file}, {right_res_file}) and probe ({probe_file})."
                    )

                left_payload = torch.load(left_res_file, map_location="cpu")
                right_payload = torch.load(right_res_file, map_location="cpu")
                probe_payload = torch.load(probe_file, map_location="cpu")
                left_residuals = left_payload["residuals"]
                right_residuals = right_payload["residuals"]
                probe_direction = probe_payload["probe_direction"]

                if STEP7_USE_GLOBAL_POLICY:
                    selected_layers = global_layers
                    layer_source = "global_best_band"
                else:
                    if not pair_policy_file.exists():
                        raise FileNotFoundError(f"Missing pair layer policy: {pair_policy_file}")
                    pair_policy = json.loads(pair_policy_file.read_text(encoding="utf-8"))
                    bb = pair_policy["best_band"]
                    selected_layers = list(range(int(bb["start"]), int(bb["end"]) + 1))
                    layer_source = "pair_best_band"

                if STEP7_TEMPERATURE_SOURCE == "global":
                    temperature = float(global_cal["best_temperature"])
                    temperature_source = "global_calibration"
                elif STEP7_TEMPERATURE_SOURCE == "pair":
                    if not pair_cal_file.exists():
                        raise FileNotFoundError(f"Missing pair calibration file: {pair_cal_file}")
                    pair_cal = json.loads(pair_cal_file.read_text(encoding="utf-8"))
                    temperature = float(pair_cal["best_temperature"])
                    temperature_source = "pair_calibration"
                else:
                    raise ValueError(f"Unsupported STEP7_TEMPERATURE_SOURCE: {STEP7_TEMPERATURE_SOURCE}")

                left_per_layer = per_layer_pair_scores(residuals=left_residuals, probe_direction=probe_direction)
                right_per_layer = per_layer_pair_scores(residuals=right_residuals, probe_direction=probe_direction)
                left_scores = aggregate_scores_by_layers(per_layer_scores=left_per_layer, selected_layers=selected_layers)
                right_scores = aggregate_scores_by_layers(per_layer_scores=right_per_layer, selected_layers=selected_layers)

                eval_row = evaluate_pair_predictions(
                    left_scores=left_scores,
                    right_scores=right_scores,
                    left_label=left,
                    right_label=right,
                    temperature=temperature,
                )

                row = {
                    "pair_key": pair_key,
                    "split": STEP7_EVAL_SPLIT,
                    "n_left": int(left_scores.shape[0]),
                    "n_right": int(right_scores.shape[0]),
                    "selected_layers": [int(x) for x in selected_layers],
                    "layer_source": layer_source,
                    "temperature": float(temperature),
                    "temperature_source": temperature_source,
                    **eval_row,
                }
                step7_report["pair_metrics"].append(row)

                pair_rows_for_csv.append(
                    {
                        "pair_key": pair_key,
                        "balanced_accuracy": row["balanced_accuracy"],
                        "avg_margin": row["avg_margin"],
                        "temperature": row["temperature"],
                        "layer_source": row["layer_source"],
                        "selected_layers": "-".join(str(x) for x in row["selected_layers"]),
                    }
                )

            except Exception as exc:  # noqa: PERF203
                step7_report["errors"].append(f"{pair_key}/{STEP7_EVAL_SPLIT}: {exc}")

    if step7_report["pair_metrics"]:
        pair_bal_acc = [float(r["balanced_accuracy"]) for r in step7_report["pair_metrics"]]
        pair_margins = [float(r["avg_margin"]) for r in step7_report["pair_metrics"]]
        step7_report["aggregate_metrics"] = {
            "n_pairs": len(step7_report["pair_metrics"]),
            "mean_pair_balanced_accuracy": float(sum(pair_bal_acc) / len(pair_bal_acc)),
            "min_pair_balanced_accuracy": float(min(pair_bal_acc)),
            "max_pair_balanced_accuracy": float(max(pair_bal_acc)),
            "mean_pair_avg_margin": float(sum(pair_margins) / len(pair_margins)),
        }

    if not step7_report["errors"]:
        for cfg in STEP7_OVERLAP_CHECKS:
            try:
                source_pair = cfg["source_pair"]
                target_pair = cfg["target_pair"]
                expected_map = cfg["expected_map"]

                if source_pair not in pair_lookup or target_pair not in pair_lookup:
                    step7_report["warnings"].append(
                        f"Skipping overlap check {source_pair} -> {target_pair}: pair not configured."
                    )
                    continue

                src_left, src_right = pair_lookup[source_pair]
                tgt_left, tgt_right = pair_lookup[target_pair]

                src_left_file = (
                    project_root / settings.results_dir / "residuals" / spec.name / source_pair / STEP7_EVAL_SPLIT / f"{src_left}.pt"
                )
                src_right_file = (
                    project_root / settings.results_dir / "residuals" / spec.name / source_pair / STEP7_EVAL_SPLIT / f"{src_right}.pt"
                )
                tgt_probe_file = (
                    project_root / settings.results_dir / "probes" / spec.name / target_pair / STEP7_EVAL_SPLIT / "probe.pt"
                )
                tgt_pair_policy_file = (
                    project_root / settings.results_dir / "layer_policy" / spec.name / target_pair / STEP7_EVAL_SPLIT / "layer_policy.json"
                )
                tgt_pair_cal_file = (
                    project_root / settings.results_dir / "calibration" / spec.name / target_pair / STEP7_EVAL_SPLIT / "temperature_calibration.json"
                )

                if not src_left_file.exists() or not src_right_file.exists() or not tgt_probe_file.exists():
                    step7_report["warnings"].append(
                        f"Skipping overlap check {source_pair} -> {target_pair}: missing files."
                    )
                    continue

                src_left_res = torch.load(src_left_file, map_location="cpu")["residuals"]
                src_right_res = torch.load(src_right_file, map_location="cpu")["residuals"]
                tgt_probe = torch.load(tgt_probe_file, map_location="cpu")["probe_direction"]

                if STEP7_USE_GLOBAL_POLICY:
                    selected_layers = global_layers
                    layer_source = "global_best_band"
                else:
                    if not tgt_pair_policy_file.exists():
                        step7_report["warnings"].append(
                            f"Skipping overlap check {source_pair} -> {target_pair}: missing pair policy."
                        )
                        continue
                    pp = json.loads(tgt_pair_policy_file.read_text(encoding="utf-8"))
                    bb = pp["best_band"]
                    selected_layers = list(range(int(bb["start"]), int(bb["end"]) + 1))
                    layer_source = "pair_best_band"

                if STEP7_TEMPERATURE_SOURCE == "global":
                    temperature = float(global_cal["best_temperature"])
                    temperature_source = "global_calibration"
                else:
                    if not tgt_pair_cal_file.exists():
                        step7_report["warnings"].append(
                            f"Skipping overlap check {source_pair} -> {target_pair}: missing pair calibration."
                        )
                        continue
                    pc = json.loads(tgt_pair_cal_file.read_text(encoding="utf-8"))
                    temperature = float(pc["best_temperature"])
                    temperature_source = "pair_calibration"

                src_left_per_layer = per_layer_pair_scores(residuals=src_left_res, probe_direction=tgt_probe)
                src_right_per_layer = per_layer_pair_scores(residuals=src_right_res, probe_direction=tgt_probe)
                src_left_scores = aggregate_scores_by_layers(
                    per_layer_scores=src_left_per_layer,
                    selected_layers=selected_layers,
                )
                src_right_scores = aggregate_scores_by_layers(
                    per_layer_scores=src_right_per_layer,
                    selected_layers=selected_layers,
                )

                overlap = evaluate_mapped_cross_confusion(
                    source_left_scores_on_target_probe=src_left_scores,
                    source_right_scores_on_target_probe=src_right_scores,
                    target_left_label=tgt_left,
                    target_right_label=tgt_right,
                    expected_label_for_source_left=expected_map["source_left_to_target"],
                    expected_label_for_source_right=expected_map["source_right_to_target"],
                    temperature=temperature,
                )
                step7_report["overlap_confusion"].append(
                    {
                        "source_pair": source_pair,
                        "target_pair": target_pair,
                        "split": STEP7_EVAL_SPLIT,
                        "layer_source": layer_source,
                        "selected_layers": [int(x) for x in selected_layers],
                        "temperature": float(temperature),
                        "temperature_source": temperature_source,
                        **overlap,
                    }
                )

            except Exception as exc:  # noqa: PERF203
                step7_report["warnings"].append(
                    f"Overlap check {cfg.get('source_pair')} -> {cfg.get('target_pair')} failed: {exc}"
                )

    eval_root = project_root / settings.results_dir / "evaluation" / spec.name / STEP7_EVAL_SPLIT
    eval_root.mkdir(parents=True, exist_ok=True)

    eval_json = eval_root / "model_evaluation.json"
    eval_json.write_text(json.dumps(step7_report, indent=2), encoding="utf-8")

    csv_path = eval_root / "pair_metrics.csv"
    fieldnames = ["pair_key", "balanced_accuracy", "avg_margin", "temperature", "layer_source", "selected_layers"]
    with csv_path.open("w", encoding="utf-8", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        for row in pair_rows_for_csv:
            writer.writerow(row)

    step7_report["artifacts"] = {
        "model_evaluation_json": str(eval_json),
        "pair_metrics_csv": str(csv_path),
    }
    step7_report["ran_evaluation"] = True

step7_report

In [ ]:
step_status["step_7_complete"] = (
    bool(step7_report.get("ran_evaluation")) and not step7_report.get("errors")
)

pass

## Step 8 - Cross-Model Comparison

Aggregate model reports into final comparison tables and bundle.

In [ ]:
try:
    from emotion.comparison import (
        build_model_summary_row,
        build_pair_rows,
        load_model_evaluation,
        write_csv,
    )
    print("Using emotion.comparison from dataset package")
except Exception as _import_err:
    print("Falling back to local Step-8 helpers due to import error:")
    print(_import_err)

    import csv
    import json

    def load_model_evaluation(*, eval_file):
        if not eval_file.exists():
            raise FileNotFoundError(f"Missing evaluation file: {eval_file}")
        return json.loads(eval_file.read_text(encoding="utf-8"))

    def build_pair_rows(*, model_name, report):
        rows = []
        for r in report.get("pair_metrics", []):
            rows.append(
                {
                    "model_name": model_name,
                    "split": report.get("split"),
                    "pair_key": r.get("pair_key"),
                    "balanced_accuracy": float(r.get("balanced_accuracy", 0.0)),
                    "avg_margin": float(r.get("avg_margin", 0.0)),
                    "temperature": float(r.get("temperature", 0.0)),
                    "layer_source": r.get("layer_source"),
                    "selected_layers": ",".join(str(x) for x in r.get("selected_layers", [])),
                    "left_mean_p_left": float(r.get("left_mean_p_left", 0.0)),
                    "right_mean_p_left": float(r.get("right_mean_p_left", 0.0)),
                }
            )
        return rows

    def build_model_summary_row(*, model_name, report):
        agg = report.get("aggregate_metrics", {}) or {}
        overlap = report.get("overlap_confusion", []) or []
        overlap_acc = [float(x.get("mapped_accuracy", 0.0)) for x in overlap] or [0.0]
        return {
            "model_name": model_name,
            "split": report.get("split"),
            "n_pairs": int(agg.get("n_pairs", 0)),
            "mean_pair_balanced_accuracy": float(agg.get("mean_pair_balanced_accuracy", 0.0)),
            "min_pair_balanced_accuracy": float(agg.get("min_pair_balanced_accuracy", 0.0)),
            "max_pair_balanced_accuracy": float(agg.get("max_pair_balanced_accuracy", 0.0)),
            "mean_pair_avg_margin": float(agg.get("mean_pair_avg_margin", 0.0)),
            "mean_overlap_mapped_accuracy": float(sum(overlap_acc) / len(overlap_acc)),
            "overlap_checks": int(len(overlap)),
            "warnings_count": int(len(report.get("warnings", []))),
            "errors_count": int(len(report.get("errors", []))),
        }

    def write_csv(*, output_file, fieldnames, rows):
        output_file.parent.mkdir(parents=True, exist_ok=True)
        with output_file.open("w", encoding="utf-8", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=fieldnames)
            writer.writeheader()
            for row in rows:
                writer.writerow(row)

In [ ]:
STEP8_COMPARISON_SPLIT = "train"
STEP8_EXPECTED_MODELS = [m.name for m in settings.models if m.enabled]

pass

In [ ]:
step8_report = {
    "validated_input": bool(step_status.get("step_7_complete", False)),
    "ran_comparison": False,
    "split": STEP8_COMPARISON_SPLIT,
    "expected_models": STEP8_EXPECTED_MODELS,
    "errors": [],
    "warnings": [],
    "models_found": [],
    "missing_models": [],
    "model_summary_rows": [],
    "pair_rows": [],
    "artifacts": {},
}

if not step8_report["validated_input"]:
    step8_report["errors"].append("Step 7 is incomplete. Run Step 7 first.")

if not step8_report["errors"]:
    import json

    eval_root = project_root / settings.results_dir / "evaluation"

    for model_name in STEP8_EXPECTED_MODELS:
        eval_file = eval_root / model_name / STEP8_COMPARISON_SPLIT / "model_evaluation.json"
        if not eval_file.exists():
            step8_report["missing_models"].append(model_name)
            continue

        try:
            report = load_model_evaluation(eval_file=eval_file)
            step8_report["models_found"].append(model_name)
            step8_report["model_summary_rows"].append(
                build_model_summary_row(model_name=model_name, report=report)
            )
            step8_report["pair_rows"].extend(
                build_pair_rows(model_name=model_name, report=report)
            )
        except Exception as exc:  # noqa: PERF203
            step8_report["errors"].append(f"{model_name}: {exc}")

    if not step8_report["models_found"]:
        step8_report["errors"].append(
            f"No model evaluation files found for split={STEP8_COMPARISON_SPLIT}."
        )

    if step8_report["model_summary_rows"]:
        summary_csv = (
            project_root
            / settings.results_dir
            / "final"
            / STEP8_COMPARISON_SPLIT
            / "model_summary.csv"
        )
        summary_fields = [
            "model_name",
            "split",
            "n_pairs",
            "mean_pair_balanced_accuracy",
            "min_pair_balanced_accuracy",
            "max_pair_balanced_accuracy",
            "mean_pair_avg_margin",
            "mean_overlap_mapped_accuracy",
            "overlap_checks",
            "warnings_count",
            "errors_count",
        ]
        write_csv(
            output_file=summary_csv,
            fieldnames=summary_fields,
            rows=step8_report["model_summary_rows"],
        )
        step8_report["artifacts"]["model_summary_csv"] = str(summary_csv)

    if step8_report["pair_rows"]:
        pair_csv = (
            project_root
            / settings.results_dir
            / "final"
            / STEP8_COMPARISON_SPLIT
            / "pair_comparison.csv"
        )
        pair_fields = [
            "model_name",
            "split",
            "pair_key",
            "balanced_accuracy",
            "avg_margin",
            "temperature",
            "layer_source",
            "selected_layers",
            "left_mean_p_left",
            "right_mean_p_left",
        ]
        write_csv(
            output_file=pair_csv,
            fieldnames=pair_fields,
            rows=step8_report["pair_rows"],
        )
        step8_report["artifacts"]["pair_comparison_csv"] = str(pair_csv)

    final_json = (
        project_root
        / settings.results_dir
        / "final"
        / STEP8_COMPARISON_SPLIT
        / "comparison_bundle.json"
    )
    final_json.parent.mkdir(parents=True, exist_ok=True)
    final_json.write_text(json.dumps(step8_report, indent=2), encoding="utf-8")
    step8_report["artifacts"]["comparison_bundle_json"] = str(final_json)

    if step8_report["missing_models"]:
        step8_report["warnings"].append(
            "Missing model evaluations for: " + ", ".join(step8_report["missing_models"])
        )

    step8_report["ran_comparison"] = True

step8_report

In [ ]:
step_status["step_8_complete"] = (
    bool(step8_report.get("ran_comparison")) and not step8_report.get("errors")
)

pass

## Step 9 - Visuals

Create final tables and charts for emotion percentages and model comparison.

In [ ]:
RUN_STEP9_VISUALS = True
STEP9_SPLIT = STEP8_COMPARISON_SPLIT if "STEP8_COMPARISON_SPLIT" in globals() else "train"
STEP9_SAVE_FIGS = True
STEP9_DPI = 180

pass

In [ ]:
step9_report = {
    "validated_input": bool(step_status.get("step_8_complete", False)),
    "ran_visuals": False,
    "split": STEP9_SPLIT,
    "errors": [],
    "warnings": [],
    "artifacts": {},
}

if not step_status.get("step_8_complete", False):
    step9_report["errors"].append("Step 8 is incomplete. Run Step 8 first.")

if RUN_STEP9_VISUALS and not step9_report["errors"]:
    import json
    import numpy as np
    import pandas as pd
    import matplotlib.pyplot as plt
    from IPython.display import Markdown, display

    try:
        import seaborn as sns
        _has_seaborn = True
        sns.set_theme(style="whitegrid")
    except Exception:
        _has_seaborn = False

    final_root = project_root / settings.results_dir / "final" / STEP9_SPLIT
    final_root.mkdir(parents=True, exist_ok=True)
    plots_root = project_root / settings.plots_dir / "final" / STEP9_SPLIT
    plots_root.mkdir(parents=True, exist_ok=True)

    bundle_path = final_root / "comparison_bundle.json"
    if not bundle_path.exists():
        step9_report["errors"].append(
            f"Missing Step-8 comparison bundle: {bundle_path}"
        )
    else:
        bundle = json.loads(bundle_path.read_text(encoding="utf-8"))
        pair_rows = bundle.get("pair_rows", [])
        summary_rows = bundle.get("model_summary_rows", [])

        if not pair_rows or not summary_rows:
            step9_report["errors"].append("Step-8 bundle has empty pair/model rows.")

    if not step9_report["errors"]:
        pair_df = pd.DataFrame(pair_rows)
        summary_df = pd.DataFrame(summary_rows)

        emotion_rows = []
        for _, r in pair_df.iterrows():
            pair_key = str(r["pair_key"])
            if "_vs_" not in pair_key:
                continue
            left, right = pair_key.split("_vs_", 1)
            left_mean = float(r["left_mean_p_left"])
            right_mean = float(r["right_mean_p_left"])

            left_pct = 100.0 * 0.5 * (left_mean + (1.0 - right_mean))
            right_pct = 100.0 - left_pct

            emotion_rows.append(
                {
                    "model_name": r["model_name"],
                    "pair_key": pair_key,
                    "emotion": left,
                    "activation_percentage": left_pct,
                }
            )
            emotion_rows.append(
                {
                    "model_name": r["model_name"],
                    "pair_key": pair_key,
                    "emotion": right,
                    "activation_percentage": right_pct,
                }
            )

        emotion_df = pd.DataFrame(emotion_rows)

        emotion_order = [
            "sad", "happy", "angry", "calm", "fear",
            "confidence", "love", "hate", "anxious", "relaxed"
        ]
        model_order = sorted(summary_df["model_name"].unique().tolist())
        emotion_df["emotion"] = pd.Categorical(emotion_df["emotion"], categories=emotion_order, ordered=True)

        emotion_pivot = (
            emotion_df
            .pivot_table(
                index="model_name",
                columns="emotion",
                values="activation_percentage",
                aggfunc="mean",
                observed=False,
            )
            .reindex(index=model_order)
            .reindex(columns=emotion_order)
        )

        emotion_long_csv = final_root / "emotion_percentages_long.csv"
        emotion_pivot_csv = final_root / "emotion_percentages_pivot.csv"
        summary_csv = final_root / "model_summary_styled_source.csv"

        emotion_df.to_csv(emotion_long_csv, index=False)
        emotion_pivot.to_csv(emotion_pivot_csv)
        summary_df.to_csv(summary_csv, index=False)

        step9_report["artifacts"].update(
            {
                "emotion_percentages_long_csv": str(emotion_long_csv),
                "emotion_percentages_pivot_csv": str(emotion_pivot_csv),
                "model_summary_source_csv": str(summary_csv),
            }
        )

        model_emotion_summary = (
            emotion_df.groupby(["model_name", "emotion"], as_index=False, observed=False)["activation_percentage"].mean()
        )
        top_emotions = (
            model_emotion_summary.sort_values(["model_name", "activation_percentage"], ascending=[True, False])
            .groupby("model_name")
            .head(3)
            .copy()
        )
        top_emotions["rank"] = top_emotions.groupby("model_name").cumcount() + 1

        top_emotions_display = (
            top_emotions[["model_name", "emotion", "activation_percentage"]]
            .sort_values(["model_name", "activation_percentage"], ascending=[True, False])
            .reset_index(drop=True)
        )
        top_emotions_display.loc[top_emotions_display["model_name"].duplicated(), "model_name"] = ""

        display(Markdown("### Emotion Activation Table"))
        display(
            emotion_pivot.style
            .background_gradient(cmap="RdYlGn", axis=None)
            .format("{:.1f}")
            .set_caption("Emotion Activation Percentage per Model")
        )

        display(Markdown("<br>"))
        display(Markdown("### Top-3 Emotions by Model"))
        display(
            top_emotions_display.style
            .background_gradient(subset=["activation_percentage"], cmap="YlOrRd")
            .format({"activation_percentage": "{:.1f}"})
            .set_caption("Top-3 Activated Emotions per Model")
        )
        display(Markdown("---"))

        display(Markdown("### Chart 1: Emotion Activation Heatmap"))
        fig1, ax1 = plt.subplots(figsize=(13, 4.5))
        if _has_seaborn:
            sns.heatmap(
                emotion_pivot,
                annot=True,
                fmt=".1f",
                cmap="Spectral",
                cbar_kws={"label": "Activation %"},
                ax=ax1,
            )
        else:
            im = ax1.imshow(emotion_pivot.values, aspect="auto", cmap="Spectral")
            ax1.set_xticks(range(len(emotion_pivot.columns)))
            ax1.set_xticklabels(emotion_pivot.columns, rotation=40, ha="right")
            ax1.set_yticks(range(len(emotion_pivot.index)))
            ax1.set_yticklabels(emotion_pivot.index)
            fig1.colorbar(im, ax=ax1, label="Activation %")
        ax1.set_title("Emotion Activation Heatmap (All Models)")
        ax1.set_xlabel("Emotion")
        ax1.set_ylabel("Model")
        fig1.tight_layout()
        plt.show()
        display(Markdown("<br><br>"))
        display(Markdown("---"))
        display(Markdown("<br>"))

        display(Markdown("### Chart 2: Emotion Comparison Bars"))
        fig2, ax2 = plt.subplots(figsize=(14, 5.5))
        plot_df = model_emotion_summary.copy()
        if _has_seaborn:
            sns.barplot(
                data=plot_df,
                x="emotion",
                y="activation_percentage",
                hue="model_name",
                palette="Set2",
                ax=ax2,
            )
        else:
            emotions = emotion_order
            x = np.arange(len(emotions))
            width = 0.8 / max(1, len(model_order))
            for i, model_name in enumerate(model_order):
                sub = plot_df[plot_df["model_name"] == model_name].set_index("emotion").reindex(emotions)
                ax2.bar(x + (i - (len(model_order) - 1) / 2) * width, sub["activation_percentage"].values, width=width, label=model_name)
            ax2.set_xticks(x)
            ax2.set_xticklabels(emotions, rotation=30, ha="right")
        ax2.set_ylim(0, 100)
        ax2.set_title("Emotion Activation Percentage by Model")
        ax2.set_xlabel("Emotion")
        ax2.set_ylabel("Activation %")
        ax2.grid(axis="y", alpha=0.25)
        ax2.legend(title="Model")
        plt.setp(ax2.get_xticklabels(), rotation=30, ha="right")
        fig2.tight_layout()
        plt.show()
        display(Markdown("<br><br>"))
        display(Markdown("---"))
        display(Markdown("<br>"))

        display(Markdown("### Chart 3: Emotion Profile Curves"))
        fig3, ax3 = plt.subplots(figsize=(13, 5.5))
        x = np.arange(len(emotion_order))
        for model_name in model_order:
            y = emotion_pivot.loc[model_name, emotion_order].values
            ax3.plot(x, y, marker="o", linewidth=2.0, label=model_name)
        ax3.set_xticks(x)
        ax3.set_xticklabels(emotion_order, rotation=30, ha="right")
        ax3.set_ylim(0, 100)
        ax3.set_title("Emotion Profile Curves Across Models")
        ax3.set_xlabel("Emotion")
        ax3.set_ylabel("Activation %")
        ax3.grid(axis="both", alpha=0.25)
        ax3.legend(title="Model")
        fig3.tight_layout()

        if STEP9_SAVE_FIGS:
            chart1 = plots_root / "emotion_activation_heatmap.png"
            chart2 = plots_root / "emotion_comparison_bars.png"
            chart3 = plots_root / "emotion_profile_curves.png"
            fig1.savefig(chart1, dpi=STEP9_DPI, bbox_inches="tight")
            fig2.savefig(chart2, dpi=STEP9_DPI, bbox_inches="tight")
            fig3.savefig(chart3, dpi=STEP9_DPI, bbox_inches="tight")
            step9_report["artifacts"].update(
                {
                    "chart_heatmap": str(chart1),
                    "chart_emotion_comparison_bars": str(chart2),
                    "chart_emotion_profile_curves": str(chart3),
                }
            )

        plt.show()
        display(Markdown("<br><br>"))

        step9_report["summary"] = {
            "n_models": int(summary_df["model_name"].nunique()),
            "n_pairs": int(pair_df["pair_key"].nunique()),
            "n_emotions": int(emotion_df["emotion"].nunique()),
        }
        step9_report["ran_visuals"] = True